pip install dijkstar 

In [11]:
from dijkstar import Graph, find_path

In [12]:
class Street:
    def __init__(self, cityFrom, cityTo, time, energy):
        self.cityFrom = cityFrom
        self.cityTo = cityTo
        self.time = time
        self.energy = energy

In [13]:
file = open("coding300/file.txt", 'r')
lines = file.read().splitlines()

streets = []
for line in lines:
    # avoiding road from city to the same one (e.g. rome to rome)
    if line[0:line.index('-')] == line[line.index('-')+1:line.index('=')]:
        continue
    streets.append(Street(line[0:line.index('-')],
                   line[line.index('-')+1:line.index('=')],
                   int(line[line.index('=')+1:line.index(',')]),
                   int(line[line.index(',')+1:])))
file.close()

In [17]:
cities = set(street.cityTo for street in streets)
bestSolution = None

class Sol:
    def __init__(self, visitedCities,  time, energy, streets):
        self.visitedCities = visitedCities
        self.streets = streets
        self.time = time
        self.energy = energy
        
    def addCity(self, street):
        self.visitedCities.append(street.cityTo)
        self.streets = [s for s in self.streets if street.cityTo != s.cityTo]
        self.time += street.time
        self.energy -= street.energy
        
    def canAddCity(self, street):
        return not (self.energy - street.energy < 0 or
                (bestSolution is not None and self.time + street.time >= bestSolution.time) or
                (self.time + street.time > 515 and 'Diagon Alley' not in self.visitedCities))
    
    def copy(self):
        return Sol(list(self.visitedCities),  self.time, self.energy, list(self.streets))

    def streetsFromLastCity(self):
        return [street for street in self.streets if street.cityFrom == self.visitedCities[-1]]

def moveToNextCity(sol):
    if len(cities) == len(sol.visitedCities):
        returnStreet = [s for s in streets if s.cityFrom == sol.visitedCities[-1] and s.cityTo == sol.visitedCities[0]][0]
        if sol.canAddCity(returnStreet):
            sol.addCity(returnStreet)
            global bestSolution
            if bestSolution is None or bestSolution.time > sol.time:
                bestSolution = sol
            print("sol: ", sol.visitedCities, sol.time, sol.energy)
        return
    
    for street in sol.streetsFromLastCity():
        if (street.cityTo == sol.visitedCities[0] or 
            #('Diagon Alley' not in sol.visitedCities and (sol.time> 515)) or 
            (street.cityTo == 'Diagon Alley' and (sol.time + street.time < 135 or sol.time + street.time > 515))):
            continue
        if sol.canAddCity(street):
            copy = sol.copy()
            copy.addCity(street)
            moveToNextCity(copy)
    

# defining startingCities
for city in cities:
    sol = Sol([city], 0, 58000, list(streets))
    if city != 'Diagon Alley':
        moveToNextCity(sol)

sol:  ['Mos Eisley', 'Camelot', 'Diagon Alley', 'Roma', 'Hobbiville', 'Metropolis', 'Atlantide', 'Paperopoli', 'Hogsmeade', 'Gotham', 'Hill Valley', 'Smallville', 'Biancavilla', 'Eldorado', 'Mos Eisley'] 6737 111
sol:  ['Mos Eisley', 'Camelot', 'Diagon Alley', 'Roma', 'Hobbiville', 'Metropolis', 'Atlantide', 'Hill Valley', 'Gotham', 'Paperopoli', 'Smallville', 'Hogsmeade', 'Eldorado', 'Biancavilla', 'Mos Eisley'] 6396 966
sol:  ['Mos Eisley', 'Camelot', 'Diagon Alley', 'Roma', 'Hobbiville', 'Metropolis', 'Atlantide', 'Hill Valley', 'Gotham', 'Paperopoli', 'Hogsmeade', 'Smallville', 'Biancavilla', 'Eldorado', 'Mos Eisley'] 5603 2272
sol:  ['Mos Eisley', 'Camelot', 'Diagon Alley', 'Roma', 'Hobbiville', 'Metropolis', 'Atlantide', 'Eldorado', 'Biancavilla', 'Smallville', 'Gotham', 'Paperopoli', 'Hogsmeade', 'Hill Valley', 'Mos Eisley'] 5318 515
sol:  ['Mos Eisley', 'Camelot', 'Diagon Alley', 'Roma', 'Hobbiville', 'Metropolis', 'Atlantide', 'Eldorado', 'Biancavilla', 'Paperopoli', 'Hogsmead

In [21]:
print(str(bestSolution.time) + "-", end="")
for city in bestSolution.visitedCities:
    print(city[0:2], end="")
print("-" + str(58000-bestSolution.energy), end="")

1723-PaCaDiHoAtHoSmMeMoHiBiElRoGoPa-57560